In [6]:
import os
import pandas as pd
import json

root_dir = '/home/work/hocheol_dir/workspace'

test_path = 'data/034_cropped_data/test.json'
train_path = 'data/034_cropped_data/train.json'
val_path = 'data/034_cropped_data/val.json'

json_data = []

with open(os.path.join(root_dir, test_path), 'r') as f:
    json_data.extend(json.load(f))

with open(os.path.join(root_dir, train_path), 'r') as f:
    json_data.extend(json.load(f))

with open(os.path.join(root_dir, val_path), 'r') as f:
    json_data.extend(json.load(f))

In [7]:
df = pd.read_csv(os.path.join(root_dir, 'data/034_cropped_data/results_250507.csv'))
print(len(df))
df.drop_duplicates(subset='filename', keep='first', inplace=True)
df.drop(df[df['label'] == 6].index, inplace=True)
df

10800


,filename,label
0,left_cheek_0000_CRS_19_01.jpg,0
1,left_cheek_0000_CRS_46_01.jpg,0
2,left_cheek_0001_CRS_19_01.jpg,1
3,left_cheek_0001_CRS_46_01.jpg,1
4,left_cheek_0002_CRS_19_01.jpg,0
...,...,...
10795,right_cheek_2693_CRS_46_01.jpg,1
10796,right_cheek_2696_CRS_19_01.jpg,2
10797,right_cheek_2696_CRS_46_01.jpg,2
10798,right_cheek_2697_CRS_19_01.jpg,0


In [8]:
df.value_counts(['label'])

label
0        3822
1        2672
2        2629
3        1146
4         420
Name: count, dtype: int64

In [9]:
df[df['label'] == 6]

,filename,label


In [10]:
# id - label 쌍 만들어야 함. (_19, _46 있어서 이거 나눠줘야해)
# 딕셔너리는 id, 19_left, 19_right, 46_left, 46_right, label 있어야 함

json_data = {}
for filename, label in df.values:
    # print(filename, label)
    user_id = filename.split('_')[2]
    direction = filename.split('_')[0]
    eq_id = filename.split('_')[-2]

    # print(direction, user_id, eq_id, label)
    if user_id not in json_data.keys():
        json_data[user_id] = {
            'id':user_id,
            '19_left': '',
            '19_right': '',
            '46_left': '',
            '46_right': '',
            'labels': []
        }
    
    json_data[user_id]['labels'].append(label)
    
    file_name = f"{eq_id}_{direction}"
    if json_data[user_id][file_name] != '':
        print(f"Duplicate file name for user {user_id}: {file_name}")
    json_data[user_id][file_name] = filename

In [11]:
num = 0
different_label = 0
num_label_not_4 = 0

data_num_label_not_4 = []
name_different_label = []


# 라벨 서로 다른거 존재(diff_same~~), 종횡비 outlier 발생(~~not_4)
for id, data in json_data.items():
    label_set = set(data['labels'])
    if len(label_set) != 1:
        # print(id, data['labels'])
        different_label += 1
        name_different_label.append(id)
    if len(data['labels']) != 4:
        num_label_not_4 += 1
        data_num_label_not_4.append(data)

print(different_label)
print(num_label_not_4)

1376
38


In [12]:
len(json_data)

2691

In [13]:
# 종횡비 outlier는 해당 쌍 제외할거고, label은 labels의 평균
for id, data in json_data.items():
    data['label'] = round(sum(data['labels'])/len(data['labels']))

new_dataset = []
for id, data in json_data.items():
    # 19 중에도 하나, 46 중에도 하나가 동시에 비어있는 경우 버리기
    if '' in [data['19_left'], data['19_right']] and '' in [data['46_left'], data['46_right']]:
        continue
    else:
        label = round(sum(data['labels']) / len(data['labels']))
        
        # 써먹을 수 있는 데이터만 남기기
        new_data = {
            'id': id,
            '19_left': data['19_left'] if data['19_right'] != '' else '',
            '19_right': data['19_right'] if data['19_left'] != '' else '',
            '46_left': data['46_left'] if data['46_right'] != '' else '',
            '46_right': data['46_right'] if data['46_left'] != '' else '',
            'label': label
        }
        new_dataset.append(new_data)

In [14]:
len(new_dataset)

2656

In [16]:
# 3명의 user 빼고는 전부 4개의 이미지 모두 살아남았음
lst = [0,0,0,0,0]
for data in new_dataset:
    i = 4
    if data['19_left'] == '':
        i -= 1
    if data['19_right'] == '':
        i -= 1
    if data['46_left'] == '':
        i -= 1
    if data['46_right'] == '':
        i -= 1
    lst[i] += 1
    if i == 2:
        print(data)
print(lst)

{'id': '0727', '19_left': '', '19_right': '', '46_left': 'left_cheek_0727_CRS_46_01.jpg', '46_right': 'right_cheek_0727_CRS_46_01.jpg', 'label': 0}
{'id': '1257', '19_left': 'left_cheek_1257_CRS_19_01.jpg', '19_right': 'right_cheek_1257_CRS_19_01.jpg', '46_left': '', '46_right': '', 'label': 4}
{'id': '1714', '19_left': '', '19_right': '', '46_left': 'left_cheek_1714_CRS_46_01.jpg', '46_right': 'right_cheek_1714_CRS_46_01.jpg', 'label': 4}
[0, 0, 3, 0, 2653]


In [19]:
new_dataset[0]
with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/origin/0512_new_dataset.json', 'w') as f:
    json.dump(new_dataset, f, indent=2, ensure_ascii=False)

In [20]:
############################ 여기서 label stratify 해서 나눈 다음 그 안에서 19, 46 나눠야 함
origin_json = None
with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/origin/0512_new_dataset.json', 'r') as f:
    origin_json = json.load(f)

origin_json[0]

{'id': '0000',
 '19_left': 'left_cheek_0000_CRS_19_01.jpg',
 '19_right': 'right_cheek_0000_CRS_19_01.jpg',
 '46_left': 'left_cheek_0000_CRS_46_01.jpg',
 '46_right': 'right_cheek_0000_CRS_46_01.jpg',
 'label': 0}

In [21]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(origin_json, test_size=0.2, stratify=[data['label'] for data in origin_json], random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=[data['label'] for data in temp_data], random_state=42)
print(len(train_data), len(val_data), len(test_data))

2124 266 266


In [22]:
train_data[0]

{'id': '1876',
 '19_left': 'left_cheek_1876_CRS_19_01.jpg',
 '19_right': 'right_cheek_1876_CRS_19_01.jpg',
 '46_left': 'left_cheek_1876_CRS_46_01.jpg',
 '46_right': 'right_cheek_1876_CRS_46_01.jpg',
 'label': 0}

In [28]:
tr_json = []
for data in train_data:
    if data['19_left'] != '':
        data_19 = {
            'id': f"{data['id']}_19",
            'left_cheek_path':f"Training/{data['id']}/{data['19_left']}",
            'right_cheek_path':f"Training/{data['id']}/{data['19_right']}",
            'label': data['label']
        }
        tr_json.append(data_19)
    if data['46_left'] != '':
        data_46 = {
            'id': f"{data['id']}_46",
            'left_cheek_path':f"Training/{data['id']}/{data['46_left']}",
            'right_cheek_path':f"Training/{data['id']}/{data['46_right']}",
            'label': data['label']
        }
        tr_json.append(data_46)

print(len(tr_json), len(train_data))
tr_json.sort(key=lambda x: x['id'])
with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_dataset/train.json', 'w') as f:
    json.dump(tr_json, f, indent=2, ensure_ascii=False)

4245 2124


In [29]:
val_json = []
for data in val_data:
    if data['19_left'] != '':
        data_19 = {
            'id': f"{data['id']}_19",
            'left_cheek_path':f"Validation/{data['id']}/{data['19_left']}",
            'right_cheek_path':f"Validation/{data['id']}/{data['19_right']}",
            'label': data['label']
        }
        val_json.append(data_19)
    if data['46_left'] != '':
        data_46 = {
            'id': f"{data['id']}_46",
            'left_cheek_path':f"Validation/{data['id']}/{data['46_left']}",
            'right_cheek_path':f"Validation/{data['id']}/{data['46_right']}",
            'label': data['label']
        }
        val_json.append(data_46)
print(len(val_json), len(val_data))
val_json.sort(key=lambda x: x['id'])
with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_dataset/val.json', 'w') as f:
    json.dump(val_json, f, indent=2, ensure_ascii=False)

532 266


In [31]:
test_json = []
for data in test_data:
    if data['19_left'] != '':
        data_19 = {
            'id': f"{data['id']}_19",
            'left_cheek_path':f"Test/{data['id']}/{data['19_left']}",
            'right_cheek_path':f"Test/{data['id']}/{data['19_right']}",
            'label': data['label']
        }
        test_json.append(data_19)
    if data['46_left'] != '':
        data_46 = {
            'id': f"{data['id']}_46",
            'left_cheek_path':f"Test/{data['id']}/{data['46_left']}",
            'right_cheek_path':f"Test/{data['id']}/{data['46_right']}",
            'label': data['label']
        }
        test_json.append(data_46)
print(len(test_json), len(test_data))
test_json.sort(key=lambda x: x['id'])
with open('/home/work/hocheol_dir/workspace/data/034_cropped_data/0512_dataset/test.json', 'w') as f:
    json.dump(test_json, f, indent=2, ensure_ascii=False)

532 266
